In [1]:
import pandas as pd
import numpy as np

In [2]:
%store -r

In [3]:
bm_raw = dfs['bm_raw_new'].copy()
bss_pca = dfs['bss_pca_raw'].copy()
pca_sellers = dfs['PCA for Sellers'].copy()
classification = df['classification_mapping'].copy()
Brand_mapping = df['Brand_mapping'].copy()
search_roin = dfs['Self serve dashboard'].copy()

In [4]:
bm_raw.columns

Index(['marketplace', 'campaign_id', 'advertiser_id', 'advertiser_name',
       'BUSINESS_ACCOUNT_ID', 'BUSINESS_ACCOUNT_NAME', 'seller_id', 'brand',
       'commodity_name', 'analytic_vertical', 'analytic_category',
       'analytic_super_category', 'lifestyle_supercategory', 'Supercategory',
       'BU', 'HL_BU', 'HL_BU_New', 'Alpha_MP', 'BMP_Tag', 'Team',
       'budget_type', 'cost_model', 'pacing', 'campaign_type', 'campaign_name',
       'status', 'start_date', 'end_date', 'commodity_id', 'Channel',
       'allocated_budget', 'gross_budget', 'uniqueviewcount', 'actioncount',
       'total_cost_x', 'addtobasketcount', 'viewcount', 'cnt_lid',
       'attr_units', 'attr_revenue', 'overburn_share', 'Actual_spend',
       'Budget_adgroup', 'ROI', 'CTR', 'CVR'],
      dtype='object')

In [5]:
bm_raw = bm_raw[bm_raw['total_cost_x']>5]

In [6]:
# bss_pca.columns

In [7]:
#  pca_sellers.columns

In [8]:
classification.columns

Index(['Super Categories_Tag', 'BU_Tag', 'Wrong Tagging', 'New Tag', 'Type',
       'Adv. Name', 'Tag', 'Remarks', 'Super Categories_HL', 'BU_HL', 'BU_HL2',
       'HL Supercat'],
      dtype='object')

In [9]:
Brand_mapping.columns

Index(['Incorrect Account Name', 'Correct Name', 'BU', 'Concatenate',
       'Advertiser Name', 'Tag', 'Ad Account ID', 'Ad Account Name',
       'Business Account ID', 'Business Account Name', 'Vertical',
       'Current SC', 'New SC'],
      dtype='object')

In [10]:
incorrect_brands = Brand_mapping['Ad Account ID'].unique()

bm_raw['New Alpha/MP'] = np.where(
    bm_raw['advertiser_id'].isin(incorrect_brands) | (bm_raw['Team'] == 'IC'),
    'IC',
    bm_raw['Alpha_MP']
)


In [11]:
bm_raw['New Alpha/MP'].value_counts()

New Alpha/MP
MP       483661
Alpha     54269
IC            1
Name: count, dtype: int64

In [12]:
bm_raw['FCFF Alpha/MP'] = np.where(bm_raw['marketplace'] == "HYPERLOCAL","HL",bm_raw['Alpha_MP'])

In [13]:
bm_raw['FCFF Alpha/MP'].value_counts()

FCFF Alpha/MP
MP       483662
Alpha     44627
HL         9642
Name: count, dtype: int64

In [14]:
brand_map = dict(zip(Brand_mapping['Vertical'].drop_duplicates(), 
                     Brand_mapping['New SC']))

sc_map = dict(zip(classification['Wrong Tagging'].drop_duplicates(), 
                  classification['New Tag']))

bm_raw = bm_raw.reset_index(drop=True)

conditions = [
    bm_raw['marketplace'] == "Grocery",
    bm_raw['analytic_vertical'].isin(brand_map.keys()),
    bm_raw['analytic_super_category'].isin(brand_map.keys())
]

choices = [
    "Grocery",
    bm_raw['analytic_vertical'].map(brand_map),
    bm_raw['analytic_super_category'].map(brand_map)
]

fallback = bm_raw['analytic_super_category'].map(sc_map).fillna(bm_raw['analytic_super_category'])

bm_raw['New Supercat'] = np.select(conditions, choices, default=fallback)


In [15]:
# print(bm_raw['New Supercat'].value_counts().to_string())


In [16]:
bm_raw['SC match FCFF'] = bm_raw['New Supercat'].isin(classification['Super Categories_Tag'])

In [17]:
bm_raw['SC match FCFF'].value_counts()

SC match FCFF
True     537477
False       454
Name: count, dtype: int64

In [18]:
sc_bu_map = dict(zip(classification['Super Categories_Tag'], classification['BU_Tag']))

bm_raw['New BU'] = bm_raw['New Supercat'].map(sc_bu_map)

In [19]:
print(bm_raw['New BU'].isnull().sum())

454


In [20]:
bm_raw['Spends'] = bm_raw['total_cost_x']

In [21]:
bm_raw['Spends'].sum()

np.float64(1823300877.2999997)

In [22]:
brand_map = dict(zip(Brand_mapping['Incorrect Account Name'], Brand_mapping['Correct Name']))

bm_raw['lookup_key'] = bm_raw['advertiser_name'].astype(str) + bm_raw['analytic_super_category'].astype(str)

cond_a = bm_raw['lookup_key'].isin(Brand_mapping['Concatenate'])
cond_b = bm_raw['analytic_super_category'].isin(Brand_mapping['BU'])

bm_raw['Brand'] = np.where(
    cond_a & cond_b,
    bm_raw['advertiser_name'].map(brand_map),
    bm_raw['brand']
)
bm_raw['Brand'] = bm_raw['Brand'].fillna(bm_raw['brand'])


In [23]:
len(bm_raw[bm_raw['Brand'] == "0"])


223

In [24]:
# bm_raw['Brand'].value_counts(normalize=True)

In [25]:
bm_raw.columns

Index(['marketplace', 'campaign_id', 'advertiser_id', 'advertiser_name',
       'BUSINESS_ACCOUNT_ID', 'BUSINESS_ACCOUNT_NAME', 'seller_id', 'brand',
       'commodity_name', 'analytic_vertical', 'analytic_category',
       'analytic_super_category', 'lifestyle_supercategory', 'Supercategory',
       'BU', 'HL_BU', 'HL_BU_New', 'Alpha_MP', 'BMP_Tag', 'Team',
       'budget_type', 'cost_model', 'pacing', 'campaign_type', 'campaign_name',
       'status', 'start_date', 'end_date', 'commodity_id', 'Channel',
       'allocated_budget', 'gross_budget', 'uniqueviewcount', 'actioncount',
       'total_cost_x', 'addtobasketcount', 'viewcount', 'cnt_lid',
       'attr_units', 'attr_revenue', 'overburn_share', 'Actual_spend',
       'Budget_adgroup', 'ROI', 'CTR', 'CVR', 'New Alpha/MP', 'FCFF Alpha/MP',
       'New Supercat', 'SC match FCFF', 'New BU', 'Spends', 'lookup_key',
       'Brand'],
      dtype='object')

# BSS PCA

In [26]:
bss_pca.columns

Index(['Campaign ID', 'Marketplace', 'BU', 'Supercategory', 'HL_BU', 'store',
       'analytic_super_category', 'Brands', 'Store Name',
       'creative_template_id', 'creative_type', 'costmodel', 'status',
       'Ad Account ID', 'Ad Account Name', 'Business Account ID',
       'Business Account Name', 'Team', 'Alpha_MP', 'BMP_Tag', 'Channel',
       'lifestyle_supercategory', 'allocated_budget', 'spend', 'views',
       'clicks', 'ppv', 'click_direct_units', 'click_indirect_units',
       'click_direct_revenue', 'click_indirect_revenue', 'Product',
       'Average CPC', 'CTR', 'ROI'],
      dtype='object')

In [27]:
# bss_pca = bss_pca[bss_pca['spend']>5].copy()
bss_pca = bss_pca.copy()

In [28]:
incorrect_brands = Brand_mapping['Advertiser Name'].unique()

bss_pca['New Alpha/MP'] = np.where(
    bss_pca['Business Account Name'].isin(incorrect_brands) | (bss_pca['Team'] == 'IC'),
    'IC',
    bss_pca['Alpha_MP']
)


In [29]:
bss_pca['New Alpha/MP'].value_counts()

New Alpha/MP
Alpha    275400
MP       119955
Name: count, dtype: int64

In [30]:
# class_map = dict(zip(classification['Wrong Tagging'], classification['New Tag']))

# conditions = [ bss_pca['BU'] == "Grocery", bss_pca['BU'] == "Mobile", bss_pca['Supercategory'] == "PetCare", bss_pca['Supercategory'].isin(class_map.keys()) ]
# choices = [ "Grocery", "Mobile", "Pets", bss_pca['Supercategory'].map(class_map) ]

# bss_pca['New Supercat'] = np.select(conditions, choices, default=bss_pca['Supercategory'])

df_class_unique = classification.drop_duplicates(subset=['Wrong Tagging'])
mapping_dict = dict(zip(df_class_unique['Wrong Tagging'], df_class_unique['New Tag']))

conditions = [
    (bss_pca['BU'] == "Grocery"),
    (bss_pca['BU'] == "Mobile"),
    (bss_pca['Supercategory'] == "PetCare")
]
choices = ["Grocery", "Mobile", "Pets"]
supercat = bss_pca['Supercategory'].map(mapping_dict).fillna(bss_pca['Supercategory'])

bss_pca['New Supercat'] = np.select(conditions, choices, default=supercat)


In [31]:
bss_pca['New Supercat'].isnull().sum()

np.int64(0)

In [32]:
bss_pca['SC match FCFF'] = bss_pca['New Supercat'].isin(classification['Super Categories_Tag'])

In [33]:
bss_pca['SC match FCFF'].value_counts()

SC match FCFF
True     385098
False     10257
Name: count, dtype: int64

In [34]:
bss_pca[bss_pca['SC match FCFF'] == False]['New Supercat'].value_counts()

New Supercat
Not Assigned    10257
Name: count, dtype: int64

In [35]:
sc_bu_map = dict(zip(classification['Super Categories_Tag'], classification['BU_Tag']))

bss_pca['New BU'] = bss_pca['New Supercat'].map(sc_bu_map)

In [36]:
print(bss_pca['New BU'].isnull().sum())

10257


In [37]:
brand_map = dict(zip(Brand_mapping['Incorrect Account Name'], Brand_mapping['Correct Name']))

bss_pca['lookup_key'] = bss_pca['Ad Account Name'].astype(str) + bss_pca['Supercategory'].astype(str)

cond_a = bss_pca['lookup_key'].isin(Brand_mapping['Concatenate'])
cond_b = bss_pca['Supercategory'].isin(Brand_mapping['BU'])

bss_pca['Brand'] = np.where(
    cond_a & cond_b,
    bss_pca['Ad Account Name'].map(brand_map),
    bss_pca['Brands']
)
bss_pca['Brand'] = bss_pca['Brand'].fillna(bss_pca['Brands'])

In [38]:
len(bss_pca[bss_pca['Brand'] == "0"])

0

In [39]:
bss_pca['FCFF Alpha/MP'] = np.where(bss_pca['Alpha_MP'] == "IC","Alpha",np.where(bss_pca['Alpha_MP'] =="3P","MP",bss_pca['Alpha_MP']))

In [40]:
bss_pca['BrandxBU'] = bss_pca['Brand'].astype(str) + bss_pca['BU'].astype(str)
lookup_map = bss_pca.drop_duplicates('BrandxBU').set_index('BrandxBU')['New Supercat'].to_dict()

bss_pca['Check'] = bss_pca['BrandxBU'].map(lookup_map)

In [41]:
bss_pca['New Supercat'] = np.where(bss_pca['SC match FCFF'] == False,np.where(bss_pca['Check'] == "Not Assigned", "Not Assigned",bss_pca['Check']),bss_pca['New Supercat'])

In [42]:
bss_pca['SC match FCFF'] = bss_pca['New Supercat'].isin(classification['Super Categories_Tag'])

In [43]:
print(round(bss_pca[bss_pca['SC match FCFF'] == False ]['spend'].sum(),1))

74165.9


# Search ROINS

In [44]:
search_roin.head()

,PURCHASE_ORDER_ID,PURCHASE_ORDER_NAME,AMOUNT,START_DATE,END_DATE,PAYMENT_CYCLE,ACCOUNT_ID,ACCOUNT_NAME,BUSINESS_ACCOUNT_ID,BUSINESS_ACCOUNT_NAME,CREATED_AT,UPDATED_AT
0,00B9OOSTS3H6,4583123632,250000.0,2026-05-06,2026-07-31,30,LKGC59M0DJX0,Chocolate,UN09UEOWCQ,MONDELEZ INDIA FOODS PRIVATE LIMITED,2026-05-06T05:38:35.000Z,2026-05-06T05:38:35.000Z
1,054V0PXFOXRQ,1135147,450000.0,2026-05-01,2026-05-31,60,CK9DU84QXO1P,ABBOTT ENSURE PO,2O2BCZL8HIJ8,ABBOTT HEALTHCARE PVT LTD,2026-05-06T10:32:41.000Z,2026-05-06T10:32:41.000Z
2,05J6Y9TJMT83,1138527,500000.0,2026-05-01,2026-05-31,60,CK9DU84QXO1P,ABBOTT ENSURE PO,2O2BCZL8HIJ8,ABBOTT HEALTHCARE PVT LTD,2026-05-06T10:32:42.000Z,2026-05-06T10:32:42.000Z
3,06MQDKEA2QGU,R026/OT/26/000034/00,30000.0,2026-05-04,2026-05-31,60,BYGK7NI1ZOVR,HE - National,org-BC6MOEUSNK,EMAMI LIMITED,2026-05-04T08:08:41.000Z,2026-05-04T08:08:41.000Z
4,0AL8KATAAU55,1135637,100000.0,2026-05-01,2026-06-30,30,5J0K75IAASBF,DIAGEO_TANQUERAY_ZENITH_PO,W2FWAGSPVPDB,TLG India Pvt Ltd,2026-05-07T07:00:50.000Z,2026-05-07T07:00:50.000Z


- Start and End of Current Month

In [45]:
search_ro = search_roin[(search_roin['START_DATE'] >= SelfServe_SOM ) & (search_roin['START_DATE'] <= SelfServe_EOM) ].copy()

In [46]:
# search_ro.head()

In [47]:
bm_raw_ro = bm_raw[bm_raw['SC match FCFF'] == True].copy()

In [48]:
bm_raw_ro['total_cost_x'].sum()

np.float64(1823162902.1299996)

In [49]:
bss_pca_ro = bss_pca[bss_pca['SC match FCFF'] == True].copy()

In [50]:
bss_pca_ro['spend'].sum()

np.float64(199859952.25199986)

- Alpha/MP

In [51]:
# alpha_mp_map_bm_raw = dict(zip(bm_raw_ro['advertiser_id'],bm_raw_ro['FCFF Alpha/MP']))
alpha_mp_map_bm_raw = bm_raw_ro.drop_duplicates('advertiser_id').set_index('advertiser_id')['FCFF Alpha/MP'].to_dict()
# alpha_mp_map_bss_pca = dict(zip(bss_pca_ro['Ad Account ID'],bm_raw_ro['FCFF Alpha/MP']))
alpha_mp_map_bss_pca = bss_pca_ro.drop_duplicates('Ad Account ID').set_index('Ad Account ID')['FCFF Alpha/MP'].to_dict()

search_ro['Alpha_MP'] = search_ro['ACCOUNT_ID'].map(alpha_mp_map_bm_raw).fillna(search_ro['ACCOUNT_ID'].map(alpha_mp_map_bss_pca))

In [52]:
print(round(search_ro.groupby('Alpha_MP')['AMOUNT'].sum()))

Alpha_MP
Alpha    1.078708e+09
HL       8.544086e+07
MP       3.649816e+08
Name: AMOUNT, dtype: float64


In [53]:
print(search_ro['AMOUNT'].sum())

1531755895.5500002


- BU

In [54]:
bu_map_bm_raw = bm_raw_ro.drop_duplicates('advertiser_id').set_index('advertiser_id')['New BU'].to_dict()
bu_map_bss_pca = bss_pca_ro.drop_duplicates('Ad Account ID').set_index('Ad Account ID')['New BU'].to_dict()

search_ro['New BU'] = search_ro['ACCOUNT_ID'].map(bu_map_bm_raw).fillna(search_ro['ACCOUNT_ID'].map(bu_map_bss_pca))

In [55]:
search_ro['New BU'].value_counts()

New BU
BGM            429
Lifestyle      137
Grocery        133
Large          114
Electronics    113
Home            33
Mobile          10
Furniture        5
Name: count, dtype: int64

- Brand


In [56]:
brand_map_bm_raw = bm_raw_ro.drop_duplicates('advertiser_id').set_index('advertiser_id')['brand'].to_dict()
brand_map_bss_pca = bss_pca_ro.drop_duplicates('Ad Account ID').set_index('Ad Account ID')['Brands'].to_dict()

search_ro['New Brand'] = search_ro['ACCOUNT_ID'].map(brand_map_bm_raw).fillna(search_ro['ACCOUNT_ID'].map(brand_map_bss_pca))

- Super Category

In [57]:
sc_map_bm_raw = bm_raw_ro.drop_duplicates('advertiser_id').set_index('advertiser_id')['New Supercat'].to_dict()
sc_map_bss_pca = bss_pca_ro.drop_duplicates('Ad Account ID').set_index('Ad Account ID')['New Supercat'].to_dict()

search_ro['New Supercategory'] = search_ro['ACCOUNT_ID'].map(sc_map_bm_raw).fillna(search_ro['ACCOUNT_ID'].map(sc_map_bss_pca))

In [58]:
# search_ro['New Supercategory'].value_counts()

In [59]:
search_ro.columns

Index(['PURCHASE_ORDER_ID', 'PURCHASE_ORDER_NAME', 'AMOUNT', 'START_DATE',
       'END_DATE', 'PAYMENT_CYCLE', 'ACCOUNT_ID', 'ACCOUNT_NAME',
       'BUSINESS_ACCOUNT_ID', 'BUSINESS_ACCOUNT_NAME', 'CREATED_AT',
       'UPDATED_AT', 'Alpha_MP', 'New BU', 'New Brand', 'New Supercategory'],
      dtype='object')

In [60]:
search_ro.to_excel("Search ROINs.xlsx",index=False)

In [61]:
%store search_ro

Stored 'search_ro' (DataFrame)
